# Week 5 Lab — Advanced Python + NumPy
**Name:** Sultan Ray · **Student ID:** 28214 · **Program:** ____________

## Setup: upload `week5_kigali_weather.xlsx` to Colab (folder icon on the left) before running.

## Part 1 · List Comprehensions

In [ ]:
temps = [24.4, 25.9, 28.6, 19.3, 26.0, 22.1]
raw = ["24.4", "25.9", "28.6", "19.3"]

# Ex 1.1: I used a list comprehension to turn every temperature into Fahrenheit.
temps_f = [t * 9/5 + 32 for t in temps]
print(temps_f)

In [ ]:
# Ex 1.2: (a) keep only temps >= 26. (b) keep temps below 24 and round them.
hot = [t for t in temps if t >= 26]
cool_rounded = [round(t) for t in temps if t < 24]
print(hot)
print(cool_rounded)

In [ ]:
# Ex 1.3: CSV values are strings, so I used float() on each one, then sum/len for the average.
nums = [float(s) for s in raw]
print(nums)
print(sum(nums) / len(nums))

## Part 2 · lambda

In [ ]:
# Ex 2.1: the score is the second item in each tuple, so the lambda picks r[1]. reverse=True puts the highest first, and [:3] gives the top 3.
results = [("Aline", 85), ("Eric", 78), ("Diane", 91), ("Samuel", 92), ("Grace", 76)]
ranked = sorted(results, key=lambda r: r[1], reverse=True)
print(ranked[:3])

In [ ]:
# Ex 2.2: max() with a lambda on p[1] finds the day with the most rain.
rain_log = [(3, 0.0), (7, 12.5), (11, 3.2), (14, 26.2), (18, 8.9)]
wettest = max(rain_log, key=lambda p: p[1])
print(wettest)

## Part 3 · NumPy Arrays

In [ ]:
# Loading cell: my file was an Excel file, so I read it with pandas and made the four arrays.
# Excel saves dates as numbers (like 46266), so I convert them to real dates here.
import numpy as np
import pandas as pd

df = pd.read_excel("week5_kigali_weather.xlsx")

d = df["date"].to_numpy()
if np.issubdtype(d.dtype, np.number):
    dates = (np.datetime64("1899-12-30") + d.astype(int).astype("timedelta64[D]")).astype(str)
else:
    dates = pd.to_datetime(d).strftime("%Y-%m-%d").to_numpy().astype(str)

temps = df["temp_c"].to_numpy(dtype=float)
rain  = df["rainfall_mm"].to_numpy(dtype=float)
hum   = df["humidity_pct"].to_numpy(dtype=float)

print(len(temps), temps[:5])
print(dates[:3])

In [ ]:
# Ex 3.1: speed test. Same job (multiply 1 million numbers by 2) with a Python loop and with NumPy.
import time
n = 1_000_000

data = list(range(n))
start = time.time()
result = [x * 2 for x in data]
loop_time = time.time() - start

arr = np.arange(n)
start = time.time()
result = arr * 2
numpy_time = time.time() - start

print(f"Python: {loop_time:.4f}s")
print(f"NumPy:  {numpy_time:.4f}s")
print(f"NumPy was about {loop_time / numpy_time:.0f}x faster")

# On my machine NumPy was roughly 4 times faster (the exact number changes a bit each run).

In [ ]:
# Ex 3.2: array methods give the stats with no loop. The mean is 24.00 °C, which matches the checkpoint.
print(f"Mean: {temps.mean():.2f} °C")
print(f"Max:  {temps.max():.2f} °C")
print(f"Min:  {temps.min():.2f} °C")
print(f"Std:  {temps.std():.2f} °C")

In [ ]:
# Ex 3.3: np.argmax gives the position of the hottest day, and I use that position to get the date.
pos = np.argmax(temps)
print(f"Hottest day: {dates[pos]} at {temps[pos]} °C")

## Part 4 · Boolean Masking

In [ ]:
# Ex 4.1: the mask is True on hot days. .sum() counts them, and temps[mask].mean() averages them.
hot_mask = temps >= 26
print(f"Hot days: {hot_mask.sum()}")
print(f"Average on hot days: {temps[hot_mask].mean():.2f} °C")

In [ ]:
# Ex 4.2: rain > 0 marks rainy days. rain.sum() is the total and rain.max() is the heaviest day.
rainy_mask = rain > 0
print(f"Rainy days: {rainy_mask.sum()}")
print(f"Total rainfall: {rain.sum():.1f} mm")
print(f"Heaviest day: {rain.max():.1f} mm")

In [ ]:
# Ex 4.3: here the mask comes from rain but the numbers come from temps.
# Rainy days were 24.14 °C and dry days 23.95 °C, so almost no difference.
# For hot AND rainy I needed & with brackets around each condition.
print(f"Rainy days avg temp: {temps[rain > 0].mean():.2f} °C")
print(f"Dry days avg temp:   {temps[rain == 0].mean():.2f} °C")

hot_and_rainy = (temps >= 26) & (rain > 0)
print(f"Hot and rainy days: {hot_and_rainy.sum()}")

print(f"Humid days (>= 70%): {(hum >= 70).sum()}")

## Part 5 · Anomalies & Trends

In [ ]:
# Ex 5.1: subtracting the mean from the whole array shows how far each day is from average.
# The mask then picks the matching dates.
anomaly = temps - temps.mean()

hot_anomalies = anomaly > 3
print("Days more than 3 °C above mean:")
print(dates[hot_anomalies])

print(f"Days more than 3 °C below mean: {(anomaly < -3).sum()}")

In [ ]:
# Ex 5.2: first 30 days vs last 30 days.
first_half = temps[:30].mean()
last_half = temps[30:].mean()

print(f"First 30 days avg: {first_half:.2f} °C")
print(f"Last 30 days avg:  {last_half:.2f} °C")

# Result: Kigali got cooler in the last 30 days, about 4.4 °C lower on average.

## Bonus · Top 5 wettest days

In [ ]:
# Bonus: zip pairs each date with its rainfall, then sorted() with a lambda ranks them.
# The for loop is only used to print.
pairs = list(zip(dates, rain))
top5 = sorted(pairs, key=lambda p: p[1], reverse=True)[:5]

for date, mm in top5:
    print(f"{date}: {mm} mm")

In [ ]:
# Extra for my report: details of the wettest day, and the correlation between temperature and rain.
pos = np.argmax(rain)
print(dates[pos], rain[pos], temps[pos])
print(np.corrcoef(temps, rain)[0, 1])

## Part 6 · Reflection

**1. What did NumPy hide, and why was the loop version still important?**

NumPy hid the running total, the counter, and the step-by-step pass through the data that I wrote by hand in Week 3. With `temps.mean()` it all happens in one call, and it runs in fast compiled code, which is why it beat the Python loop in the speed test. Learning the loop first was still important because it showed me what "average" really does, so I can understand the result and debug it when something is wrong. For example, my dates came out as Excel numbers like 46272, and I could only fix that because I understood what the data was before NumPy processed it.

**2. One insight about Kigali's weather**

Rain did not cool Kigali in this period. Rainy days averaged 24.14 °C and dry days 23.95 °C, a difference of less than 0.2 °C. The correlation between temperature and rainfall was about 0.09, and the wettest day (7 September, 26.2 mm) was also one of the hottest at 27.6 °C. The real change was seasonal: the first 30 days averaged 26.21 °C and the last 30 averaged 21.78 °C, about 4.4 °C cooler. This covers only 60 days of one season, so it describes this period and not Kigali's climate in general.

## Short Report

I analysed 60 days of Kigali weather data (September to October 2026) using Python and NumPy, with no loops in Parts 3 to 5.

**Key insight: rain did not cool Kigali in this period.** Rainy days averaged 24.14 °C and dry days 23.95 °C. The correlation between temperature and rainfall was about 0.09, close to zero. The wettest day, 7 September (26.2 mm), was also one of the hottest at 27.6 °C.

**What did change was the season.** The first 30 days averaged 26.21 °C and the last 30 averaged 21.78 °C, about 4.4 °C cooler. There were 18 hot days (≥ 26 °C, average 27.11 °C) and 16 rainy days, with 331.2 mm of total rainfall. The hottest day was 17 September at 28.6 °C.

**Limits:** this is only 60 days from one season, so the results describe this period and say nothing about Kigali's long-term climate.